In [1]:
import pandas as pd

df_ground_truth = pd.read_csv("data/ground_truth-new.csv")
ground_truth = df_ground_truth.to_dict(orient="records")

In [2]:
from ingest import load_faq_data, build_index

documents = load_faq_data()

documents_llm = []

for doc in documents:
    if doc["course"] == "llm-zoomcamp":
        documents_llm.append(doc)

documents = documents_llm
index = build_index(documents)

In [3]:
ground_truth[10]

{'question': 'How do I join the Office Hours or live workshop if I don’t have the Zoom link?',
 'document': '489dd1c9d9'}

In [4]:
doc_idx = {}

for doc in documents:
    doc_idx[doc["id"]] = doc

In [5]:
q = ground_truth[10]
q

{'question': 'How do I join the Office Hours or live workshop if I don’t have the Zoom link?',
 'document': '489dd1c9d9'}

In [6]:
doc_idx[q['document']]

{'id': '489dd1c9d9',
 'course': 'llm-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'What is the video/zoom link to the stream for the “Office Hours” or live/workshop sessions?',
 'answer': 'The zoom link is only published to instructors/presenters/TAs.\n\nStudents participate via YouTube Live and submit questions to Slido (link is pinned in the chat when live). The video URL should be posted in the [announcements channel on Telegram and Slack](https://t.me/dezoomcamp) before it begins. You can also watch live on the DataTalksClub [YouTube Channel](https://www.youtube.com/c/DataTalksClub).\n\nDon’t post questions in chat as they may be missed if the room is very active.'}

In [7]:
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv('llm.env')
openai_client = OpenAI()

In [8]:
from evaluation_utils import RAGWithUsage

assistant = RAGWithUsage(
    index=index,
    llm_client=openai_client,
)

In [9]:
rec = ground_truth[0]
question = rec["question"]

answer_llm = assistant.rag(question)
answer_llm

'Yes, it’s okay to join late. If you want a certificate, make sure you submit your project while submissions are still open.'

In [10]:
assistant.total_cost()

0.00048300000000000003

In [11]:
doc_id = rec["document"]
original_doc = doc_idx[doc_id]
answer_orig = original_doc["answer"]

answer_orig

'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'

In [12]:
rag_result = {
    "question": question,
    "answer_llm": answer_llm,
    "answer_orig": answer_orig,
    "document": doc_id,
}

rag_result

{'question': 'Is it okay to join the course late if I just found it now?',
 'answer_llm': 'Yes, it’s okay to join late. If you want a certificate, make sure you submit your project while submissions are still open.',
 'answer_orig': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.',
 'document': '74eb249bbf'}

In [13]:
def generate_rag_answer(rec):
    question = rec["question"]
    doc_id = rec["document"]
    original_doc = doc_idx[doc_id]

    answer_llm = assistant.rag(question)
    answer_orig = original_doc["answer"]

    result = {
        "question": question,
        "answer_llm": answer_llm,
        "answer_orig": answer_orig,
        "document": doc_id,
    }

    return result

In [14]:
answer_record = generate_rag_answer(ground_truth[0])
answer_record

{'question': 'Is it okay to join the course late if I just found it now?',
 'answer_llm': 'Yes, but if you want to receive a certificate, you need to submit your project while they’re still accepting submissions.',
 'answer_orig': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.',
 'document': '74eb249bbf'}

In [15]:
assistant.reset_usage()

In [16]:
from concurrent.futures import ThreadPoolExecutor
from evaluation_utils import map_progress

In [17]:
ground_truth_filtered = [rec for rec in ground_truth if rec["document"] in doc_idx]
print(f"Kept {len(ground_truth_filtered)} of {len(ground_truth)} records")

with ThreadPoolExecutor(max_workers=6) as pool:
    results = map_progress(pool, ground_truth_filtered, generate_rag_answer)

Kept 315 of 395 records


  0%|          | 0/315 [00:00<?, ?it/s]

In [18]:
answers = []

for answer_record in results:
    answers.append(answer_record)

In [19]:
assistant.total_cost()

0.31292775000000006

In [20]:
df_answers = pd.DataFrame(answers)
df_answers.to_csv("data/rag-answers-new.csv", index=False)

In [21]:
from pydantic import BaseModel, Field
from typing import Literal

class AnswerEvaluation(BaseModel):
    reasoning: str = Field(
        description="Reasoning about the quality of the answer."
    )
    score: Literal["good", "bad"] = Field(
        description="'good' if the answer is correct and complete, 'bad' otherwise."
    )

In [22]:
aqa_judge_instructions = """
You are an expert evaluator. You will be given:
1. A question from a student
2. The original answer from the FAQ (ground truth)
3. An answer generated by an AI assistant

Your task is to decide if the AI answer is semantically equivalent to
the original answer.

Rules:
- The AI answer does NOT need to be word-for-word identical
- It should convey the same key information
- Extra detail is fine as long as the core answer is correct
- Mark 'bad' only if the AI answer is wrong or misses the key point

Be fair and focus on correctness, not style.
""".strip()

In [23]:
aqa_judge_prompt = """
Question:
{question}

Original Answer (ground truth):
{answer_orig}

AI Answer:
{answer_llm}
""".strip()

In [24]:
from dotenv import load_dotenv
from openai import OpenAI
from evaluation_utils import calc_price, calc_total_price, llm_structured_retry, map_progress

load_dotenv('llm.env')
openai_client = OpenAI()

In [25]:
rec = answers[0]
rec

{'question': 'Is it okay to join the course late if I just found it now?',
 'answer_llm': 'Yes, you can still join the course late. If you want a certificate, make sure you submit your project while submissions are still open.',
 'answer_orig': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.',
 'document': '74eb249bbf'}

In [26]:
prompt = aqa_judge_prompt.format(
    question=rec["question"],
    answer_orig=rec["answer_orig"],
    answer_llm=rec["answer_llm"]
)

print(prompt)

Question:
Is it okay to join the course late if I just found it now?

Original Answer (ground truth):
Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.

AI Answer:
Yes, you can still join the course late. If you want a certificate, make sure you submit your project while submissions are still open.


In [27]:
eval_result, usage = llm_structured_retry(
    openai_client,
    aqa_judge_instructions,
    prompt,
    AnswerEvaluation,
)

print(eval_result)

reasoning='The AI answer preserves the key meaning of the ground truth: late joining is allowed, and certificate eligibility depends on submitting the project while submissions are still open. It is semantically equivalent.' score='good'


In [28]:
calc_price(usage)

{'input_cost': 0.0002205, 'output_cost': 0.0002385, 'total_cost': 0.000459}

In [29]:
def evaluate_aqa(question, answer_orig, answer_llm, model="gpt-5.4-mini"):
    prompt = aqa_judge_prompt.format(
        question=question,
        answer_orig=answer_orig,
        answer_llm=answer_llm
    )

    result, usage = llm_structured_retry(
        openai_client,
        aqa_judge_instructions,
        prompt,
        AnswerEvaluation,
        model=model,
    )

    return result, usage

In [30]:
eval_result, usage = evaluate_aqa(
    question=rec["question"],
    answer_orig=rec["answer_orig"],
    answer_llm=rec["answer_llm"]
)

eval_result

AnswerEvaluation(reasoning='The AI answer preserves the key points of the ground truth: late joining is allowed, and certificate eligibility depends on submitting the project before submissions close. This is semantically equivalent.', score='good')

In [31]:
def judge_record(rec):
    eval_result, usage = evaluate_aqa(
        question=rec["question"],
        answer_orig=rec["answer_orig"],
        answer_llm=rec["answer_llm"]
    )

    result = {
        "question": rec["question"],
        "document": rec["document"],
        "score": eval_result.score,
        "reasoning": eval_result.reasoning,
    }

    return result, usage

In [32]:
with ThreadPoolExecutor(max_workers=6) as pool:
    results = map_progress(pool, answers, judge_record)

  0%|          | 0/315 [00:00<?, ?it/s]

In [33]:
evaluations = []
usages = []

for evaluation, usage in results:
    evaluations.append(evaluation)
    usages.append(usage)

In [34]:
evaluations

[{'question': 'Is it okay to join the course late if I just found it now?',
  'document': '74eb249bbf',
  'score': 'good',
  'reasoning': 'The AI answer preserves the key meaning of the ground truth: late enrollment is allowed, and certificate eligibility depends on submitting the project before submissions close. This is semantically equivalent.'},
 {'question': 'Can I still take this course even if I missed the start date?',
  'document': '74eb249bbf',
  'score': 'good',
  'reasoning': 'The AI answer matches the ground truth: it says you can still join after the start date and correctly notes that to receive a certificate, the project must be submitted while submissions are still accepted. Extra phrasing about course materials being available does not change the meaning.'},
 {'question': 'If I join after the course has already started, am I still eligible for a certificate?',
  'document': '74eb249bbf',
  'score': 'good',
  'reasoning': 'The AI answer preserves the core point: joinin

In [35]:
usages

[ResponseUsage(input_tokens=294, input_tokens_details=InputTokensDetails(cached_tokens=0, cache_write_tokens=0), output_tokens=51, output_tokens_details=OutputTokensDetails(reasoning_tokens=0), total_tokens=345),
 ResponseUsage(input_tokens=312, input_tokens_details=InputTokensDetails(cached_tokens=0, cache_write_tokens=0), output_tokens=68, output_tokens_details=OutputTokensDetails(reasoning_tokens=0), total_tokens=380),
 ResponseUsage(input_tokens=314, input_tokens_details=InputTokensDetails(cached_tokens=0, cache_write_tokens=0), output_tokens=84, output_tokens_details=OutputTokensDetails(reasoning_tokens=0), total_tokens=398),
 ResponseUsage(input_tokens=285, input_tokens_details=InputTokensDetails(cached_tokens=0, cache_write_tokens=0), output_tokens=53, output_tokens_details=OutputTokensDetails(reasoning_tokens=0), total_tokens=338),
 ResponseUsage(input_tokens=336, input_tokens_details=InputTokensDetails(cached_tokens=0, cache_write_tokens=0), output_tokens=100, output_tokens_de

In [36]:
calc_total_price(usages)

0.2044177500000002

In [37]:
df_eval = pd.DataFrame(evaluations)

In [38]:
df_eval.head()

,question,document,score,reasoning
0,Is it okay to join the course late if I just f...,74eb249bbf,good,The AI answer preserves the key meaning of the...
1,Can I still take this course even if I missed ...,74eb249bbf,good,The AI answer matches the ground truth: it say...
2,If I join after the course has already started...,74eb249bbf,good,The AI answer preserves the core point: joinin...
3,Do I need to submit my project before submissi...,74eb249bbf,good,The AI answer matches the ground truth: it say...
4,I’m a bit late to the course—what do I need to...,74eb249bbf,good,The AI answer preserves the core requirement t...


In [39]:
df_eval.score.value_counts()

score
good    290
bad      25
Name: count, dtype: int64

In [40]:
df_eval.score.value_counts(normalize=True)

score
good    0.920635
bad     0.079365
Name: proportion, dtype: float64

In [41]:
good_count = (df_eval["score"] == "good").sum()
total_count = len(df_eval)
print(f"Good: {good_count}/{total_count} = {good_count/total_count:.2%}")

Good: 290/315 = 92.06%


In [42]:
df_eval[df_eval["score"] == "bad"].head()

,question,document,score,reasoning
24,Is peer-review of the capstone project require...,69d122f12e,bad,The AI answer says peer-reviewing 3 capstone p...
27,What do I actually need to pass in order to ge...,9f689c185f,bad,The AI answer correctly states that the Capsto...
39,Can you tell me when the course is coming back?,bd31146b0e,bad,The ground truth gives a specific return time:...
58,Is the `OpenAI API Error: Insufficient credits...,f5df151c59,bad,The ground truth says the error can occur when...
62,I added billing but still get an insufficient_...,152af39a53,bad,The AI answer captures only one minor part of ...


In [43]:
df_eval.to_csv("data/rag-evaluations-new.csv", index=False)

In [44]:
from dotenv import load_dotenv
from openai import OpenAI
from toyaikit.llm import OpenAIClient

In [45]:
def search(query: str) -> list[dict]:
    """
    Search the FAQ database for entries matching the given query.
    """
    return index.search(
        query,
        num_results=5,
        boost_dict={"question": 1.0, "answer": 2.0, "section": 0.1},
        filter_dict={"course": "llm-zoomcamp"}
    )

In [46]:
from toyaikit.tools import Tools
from toyaikit.chat.runners import OpenAIResponsesRunner

agent_tools = Tools()
agent_tools.add_tool(search)

instructions = """
You're a course teaching assistant. Answer student questions based on
the FAQ search results. Use the search tool before answering.
""".strip()

runner = OpenAIResponsesRunner(
    tools=agent_tools,
    developer_prompt=instructions,
    llm_client=OpenAIClient(model="gpt-5.4-mini")
)

In [47]:
rec = ground_truth[0]

result = runner.loop(prompt=rec["question"])

In [48]:
result.all_messages

[EasyInputMessage(content="You're a course teaching assistant. Answer student questions based on\nthe FAQ search results. Use the search tool before answering.", role='developer', phase=None, type=None),
 EasyInputMessage(content='Is it okay to join the course late if I just found it now?', role='user', phase=None, type=None),
 ResponseFunctionToolCall(arguments='{"query":"late enrollment join course late found it now" }', call_id='call_Sz6oyl6UMwFwPYmbrg0nx8b4', name='search', type='function_call', id='fc_07ed0c069f2def70006a513b17435481928624e0bb52f820b3', namespace=None, status='completed'),
 {'type': 'function_call_output',
  'call_id': 'call_Sz6oyl6UMwFwPYmbrg0nx8b4',
  'output': '[\n  {\n    "id": "74eb249bbf",\n    "course": "llm-zoomcamp",\n    "section": "General Course-Related Questions",\n    "question": "I just discovered the course. Can I still join?",\n    "answer": "Yes, but if you want to receive a certificate, you need to submit your project while we\\u2019re still acc

In [49]:
def extract_tool_calls(messages):
    tool_calls = []

    for message in messages:
        if isinstance(message, dict):
            continue

        if message.type == "function_call":
            tool_calls.append({
                "name": message.name,
                "arguments": message.arguments,
            })

    return tool_calls

In [50]:
tool_calls = extract_tool_calls(result.all_messages)

tool_calls

[{'name': 'search',
  'arguments': '{"query":"late enrollment join course late found it now" }'}]

In [51]:
doc_id = rec["document"]
original_doc = doc_idx[doc_id]
answer_orig = original_doc["answer"]

In [52]:
agent_result = {
    "question": rec["question"],
    "answer_agent": result.last_message,
    "answer_orig": answer_orig,
    "tool_calls": tool_calls,
    "cost": result.cost.total_cost,
    "document": doc_id,
}

agent_result

{'question': 'Is it okay to join the course late if I just found it now?',
 'answer_agent': 'Yes — you can still join the course now.\n\nAccording to the FAQ, if you just discovered the course, you can still start it. The only caveat is that if you want a certificate, you need to submit your project while submissions are still open.',
 'answer_orig': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.',
 'tool_calls': [{'name': 'search',
   'arguments': '{"query":"late enrollment join course late found it now" }'}],
 'cost': Decimal('0.00111525'),
 'document': '74eb249bbf'}

In [53]:
def generate_agent_answer(rec):
    doc_id = rec["document"]
    original_doc = doc_idx[doc_id]

    result = runner.loop(prompt=rec["question"])

    tool_calls = extract_tool_calls(result.all_messages)

    answer_record = {
        "question": rec["question"],
        "answer_agent": result.last_message,
        "answer_orig": original_doc["answer"],
        "tool_calls": tool_calls,
        "cost": result.cost.total_cost,
        "document": doc_id,
    }

    return answer_record

In [54]:
ground_truth_filtered = [rec for rec in ground_truth if rec["document"] in doc_idx]

with ThreadPoolExecutor(max_workers=6) as pool:
    agent_answers = map_progress(pool, ground_truth_filtered[:50], generate_agent_answer)

  0%|          | 0/50 [00:00<?, ?it/s]

In [55]:
df_agent = pd.DataFrame(agent_answers)

In [56]:
df_agent["cost"].sum()

Decimal('0.06532275')

In [57]:
df_agent.to_csv("data/agent-answers.csv", index=False)

In [58]:
df_agent = pd.read_csv("data/agent-answers.csv")
agent_answers = df_agent.to_dict(orient="records")

In [59]:
from pydantic import BaseModel, Field
from typing import Literal

class AgentEvaluation(BaseModel):
    answer_reasoning: str = Field(
        description="Reasoning about whether the final answer is correct."
    )
    answer_score: Literal["good", "bad"] = Field(
        description="'good' if the final answer matches the original answer."
    )
    trajectory_reasoning: str = Field(
        description="Reasoning about whether the tool calls were useful."
    )
    trajectory_score: Literal["good", "bad"] = Field(
        description="'good' if the tool calls were reasonable for the question."
    )